# Advanced Behavioral Biometric Feature Analysis (V4) - Comprehensive Feature Extraction

This notebook implements the **Complete Feature Dictionary** for Keystroke Dynamics. It extracts statistical moments, parametric distribution fits, and general metadata for **Dwell Times** and **Digraph Latencies**.

### Feature Categories:
1. **Metadata**: User ID, Trial ID, Label.
2. **Dwell/Digraph Parametric**: Gamma and Log-Normal parameters + Best Fit.
3. **Moments/Stats**: Mean, Std, Skewness, Kurtosis.
4. **General**: Sample count, Timestamp, and Rhythm Entropy.

**Scope**: First 5 Users, 'True' Trials only.

In [1]:
import json
import os
import numpy as np
import pandas as pd
from scipy import stats
from scipy.stats import gamma, lognorm, skew, kurtosis, entropy
import datetime

import warnings
warnings.filterwarnings('ignore')

# Configuration
RAW_DATA_PATH = r'../../data_raw/feature_kmt_dataset_Edge_Hill_University_22/feature_kmt_json';
USERS_TO_LOAD = 88;
OUTPUT_FILE = '../../data_processed/feature_kmt_dataset_Edge_Hill_University_22/feature_extraction_V4/feature_extraction_V4_88_users.csv'

## 1. Core Extraction Functions

We extract Dwell Times (Hold duration) and Digraph Latencies (Inter-key duration).

In [2]:
def get_raw_latencies(events):
    """
    Extracts raw Dwell Times and Digraph Latencies from key events.
    """
    last_press = {}
    raw_dwells = []
    raw_digraphs = []
    last_press_time = None
    
    for event in events:
        key = event.get('Key')
        event_type = event.get('Event')
        try:
            epoch = float(event.get('Epoch'))
        except (ValueError, TypeError):
            continue
            
        if event_type == 'pressed':
            last_press[key] = epoch
            if last_press_time is not None:
                latency = epoch - last_press_time
                if 0 < latency < 5: # Filter unreasonable values
                    raw_digraphs.append(latency)
            last_press_time = epoch
        elif event_type == 'released':
            if key in last_press:
                dwell = epoch - last_press[key]
                if 0 < dwell < 2:
                    raw_dwells.append(dwell)
                del last_press[key]
                
    return raw_dwells, raw_digraphs

def remove_outliers(data):
    """
    Removes outliers using the IQR method.
    """
    if len(data) < 5: return np.array(data)
    q1, q3 = np.percentile(data, [25, 75])
    iqr = q3 - q1
    lower_bound = q1 - (1.5 * iqr)
    upper_bound = q3 + (1.5 * iqr)
    return np.array([x for x in data if max(0.001, lower_bound) <= x <= upper_bound])

## 2. Statistical and Parametric Fitting

This function calculates moments, fits distributions, and computes entropy.

In [3]:
def calculate_features(data, prefix):
    """
    Calculates a full set of features for a given latency array.
    """
    features = {}
    
    # Default NaN values for all features
    feat_names = ['gamma_shape', 'gamma_loc', 'gamma_scale', 
                  'lognorm_s', 'lognorm_loc', 'lognorm_scale', 'best_fit',
                  'moment_mean', 'moment_std', 'moment_skew', 'moment_kurtosis']
    for feat in feat_names:
        features[f"{prefix}_{feat}"] = np.nan

    if len(data) < 5:
        return features

    # 1. Moments/Stats
    try:
        features[f"{prefix}_moment_mean"] = np.mean(data)
        features[f"{prefix}_moment_std"] = np.std(data)
        features[f"{prefix}_moment_skew"] = skew(data)
        features[f"{prefix}_moment_kurtosis"] = kurtosis(data)
    except Exception:
        pass

    # 2. Distribution Fitting
    # Ensure data is strictly positive for Gamma and Log-Normal with floc=0
    data_fit = np.array([x for x in data if x > 0])
    if len(data_fit) < 5: return features
    
    g_loglik = -np.inf
    l_loglik = -np.inf

    # Gamma Fit
    try:
        g_shape, g_loc, g_scale = gamma.fit(data_fit, floc=0)
        g_loglik = np.sum(gamma.logpdf(data_fit, g_shape, loc=g_loc, scale=g_scale))
        features[f"{prefix}_gamma_shape"] = g_shape
        features[f"{prefix}_gamma_loc"] = g_loc
        features[f"{prefix}_gamma_scale"] = g_scale
    except Exception:
        pass

    # Log-Normal Fit
    try:
        l_s, l_loc, l_scale = lognorm.fit(data_fit, floc=0)
        l_loglik = np.sum(lognorm.logpdf(data_fit, l_s, loc=l_loc, scale=l_scale))
        features[f"{prefix}_lognorm_s"] = l_s
        features[f"{prefix}_lognorm_loc"] = l_loc
        features[f"{prefix}_lognorm_scale"] = l_scale
    except Exception:
        pass

    # Best Fit Determination
    if not np.isinf(g_loglik) or not np.isinf(l_loglik):
        features[f"{prefix}_best_fit"] = 'Gamma' if g_loglik > l_loglik else 'Log-Normal'

    return features

def calculate_entropy(data, bins=10):
    """
    Calculates the Shannon Entropy of the binned latency distribution.
    """
    if len(data) < 2: return 0
    hist, _ = np.histogram(data, bins=bins, density=True)
    return entropy(hist)

## 3. Main Extraction Pipeline

Processing the first 5 users, focusing on 'True' trials.

In [4]:
all_records = []
file_list = sorted([f for f in os.listdir(RAW_DATA_PATH) if f.endswith('.json')])[:USERS_TO_LOAD]

print(f"Processing {len(file_list)} users...")

for file_name in file_list:
    user_id = file_name.split('_')[-1].replace('.json', '')
    file_path = os.path.join(RAW_DATA_PATH, file_name)
    
    with open(file_path, 'r') as f:
        data = json.load(f)
        
        # Process both 'true_data' and 'false_data' trials
        for trial_type, label in [('true_data', 1), ('false_data', 0)]:
            trials = data.get(trial_type, {})
        
            for trial_id, trial_content in trials.items():
                key_events = trial_content.get('key_events', [])
                if not key_events: continue
                
                # 1. Extract Timings
                raw_dwells, raw_digraphs = get_raw_latencies(key_events)
                
                # 2. Clean Data
                dwells = remove_outliers(raw_dwells)
                digraphs = remove_outliers(raw_digraphs)
                
                # 3. Calculate Features
                combined_timings = np.concatenate([dwells, digraphs]) if len(dwells) > 0 and len(digraphs) > 0 else np.array([])
                
                global_gamma_shape = np.nan
                global_gamma_scale = np.nan
                if len(combined_timings) >= 5:
                    try:
                        # Ensure strictly positive for floc=0
                        combined_fit = combined_timings[combined_timings > 0]
                        if len(combined_fit) >= 5:
                            g_shape, _, g_scale = gamma.fit(combined_fit, floc=0)
                            global_gamma_shape = g_shape
                            global_gamma_scale = g_scale
                    except Exception:
                        pass
    
                record = {
                    'user_id': user_id,
                    'trial_id': trial_id,
                    'label': label, # 1 for True trial, 0 for False trial
                    'n_samples': len(key_events),
                    'timestamp': key_events[0].get('Timestamp', 'N/A'),
                    'entropy': calculate_entropy(digraphs), # Rhythm entropy from digraphs
                    'gamma_shape': global_gamma_shape,
                    'gamma_scale': global_gamma_scale
                }
                
                # Add Dwell features
                record.update(calculate_features(dwells, 'dwell'))
                
                # Add Digraph features
                record.update(calculate_features(digraphs, 'digraph'))
                
                all_records.append(record)
    
df_final = pd.DataFrame(all_records)
print(f"Extraction complete. Total records: {len(df_final)}")
display(df_final.head())

Processing 88 users...
Extraction complete. Total records: 1760


,user_id,trial_id,label,n_samples,timestamp,entropy,gamma_shape,gamma_scale,dwell_gamma_shape,dwell_gamma_loc,...,digraph_gamma_loc,digraph_gamma_scale,digraph_lognorm_s,digraph_lognorm_loc,digraph_lognorm_scale,digraph_best_fit,digraph_moment_mean,digraph_moment_std,digraph_moment_skew,digraph_moment_kurtosis
0,0001,test_1,1,118,2022-02-06 18:57:09.981740,1.815558,3.508623,0.033074,27.332493,0,...,0,0.059946,0.699024,0,0.119396,Gamma,0.148146,0.097867,1.428131,1.902093
1,0001,test_2,1,104,2022-02-06 18:57:29.484559,1.993208,7.624057,0.013270,36.094959,0,...,0,0.022182,0.454099,0,0.107135,Gamma,0.118036,0.050511,0.619407,-0.146840
2,0001,test_3,1,104,2022-02-06 18:57:46.676618,1.936614,6.231205,0.016811,24.226644,0,...,0,0.028205,0.494156,0,0.110100,Log-Normal,0.123904,0.061066,1.123256,1.476727
3,0001,test_4,1,92,2022-02-06 18:58:08.869032,1.964406,6.028435,0.017774,27.260723,0,...,0,0.029913,0.491927,0,0.114015,Log-Normal,0.128646,0.065753,1.196697,1.292244
4,0001,test_5,1,94,2022-02-06 18:58:24.871896,1.788424,5.225757,0.019040,21.852163,0,...,0,0.033731,0.557920,0,0.102616,Log-Normal,0.119025,0.064363,0.980636,1.041489


## 4. Save and Verify

Ensuring the output follows the required dictionary format.

In [5]:
if not os.path.exists(os.path.dirname(OUTPUT_FILE)):
    os.makedirs(os.path.dirname(OUTPUT_FILE))

df_final.to_csv(OUTPUT_FILE, index=False)
print(f"File saved to: {OUTPUT_FILE}")

# Show columns to verify against dictionary
print("\nExtracted Columns:")
for col in df_final.columns:
    print(f"- {col}")

File saved to: ../../data_processed/feature_kmt_dataset_Edge_Hill_University_22/feature_extraction_V4/feature_extraction_V4_88_users.csv

Extracted Columns:
- user_id
- trial_id
- label
- n_samples
- timestamp
- entropy
- gamma_shape
- gamma_scale
- dwell_gamma_shape
- dwell_gamma_loc
- dwell_gamma_scale
- dwell_lognorm_s
- dwell_lognorm_loc
- dwell_lognorm_scale
- dwell_best_fit
- dwell_moment_mean
- dwell_moment_std
- dwell_moment_skew
- dwell_moment_kurtosis
- digraph_gamma_shape
- digraph_gamma_loc
- digraph_gamma_scale
- digraph_lognorm_s
- digraph_lognorm_loc
- digraph_lognorm_scale
- digraph_best_fit
- digraph_moment_mean
- digraph_moment_std
- digraph_moment_skew
- digraph_moment_kurtosis


## 5. Mouse Dynamics and Flight Time Features

This part adds Mouse Dynamics (Trajectory, Interaction, Directional) and additional Keystroke Flight Time variations (PP, RR).

In [6]:
def get_flight_times(events):
    """
    Extracts PP (Press-Press) and RR (Release-Release) latencies.
    """
    pp_latencies = []
    rr_latencies = []
    last_press_time = None
    last_release_time = None
    
    for event in events:
        event_type = event.get('Event')
        try:
            epoch = float(event.get('Epoch'))
        except (ValueError, TypeError):
            continue
            
        if event_type == 'pressed':
            if last_press_time is not None:
                pp = epoch - last_press_time
                if 0 < pp < 5: pp_latencies.append(pp)
            last_press_time = epoch
        elif event_type == 'released':
            if last_release_time is not None:
                rr = epoch - last_release_time
                if 0 < rr < 5: rr_latencies.append(rr)
            last_release_time = epoch
            
    return pp_latencies, rr_latencies

def get_mouse_metrics(mouse_events):
    """
    Extracts a wide range of mouse trajectory and interaction features.
    """
    click_dwells = []
    pause_times = []
    velocities = []
    accelerations = []
    curvatures = []
    path_lengths = []
    angles = []
    
    press_time = None
    last_event_time = None
    movements = {}
    
    if not mouse_events: return {}

    # Filter out events without Epoch and sort
    mouse_events = [e for e in mouse_events if 'Epoch' in e]
    if not mouse_events: return {}

    mouse_events = sorted(mouse_events, key=lambda x: float(x.get('Epoch', 0)))
    
    try:
        duration = float(mouse_events[-1]['Epoch']) - float(mouse_events[0]['Epoch'])
    except (ValueError, TypeError, KeyError):
        duration = 0

    click_count = 0
    
    for event in mouse_events:
        etype = event.get('Event')
        epoch = float(event.get('Epoch'))
        coords = event.get('Coordinates') # [x, y]
        mid = event.get('Movement ID')
        
        if last_event_time is not None:
            pause = epoch - last_event_time
            if pause > 0: pause_times.append(pause)
        
        if etype == 'left press':
            press_time = epoch
            click_count += 1
        elif etype == 'left release':
            if press_time is not None:
                click_dwells.append(epoch - press_time)
                press_time = None
        elif etype == 'movement':
            if mid is not None:
                if mid not in movements: movements[mid] = []
                movements[mid].append((epoch, coords))
        
        last_event_time = epoch
        
    # Process Movement Segments
    for mid, pts in movements.items():
        if len(pts) < 2: continue
        
        total_dist = 0
        seg_velocities = []
        for i in range(1, len(pts)):
            t1, (x1, y1) = pts[i-1]
            t2, (x2, y2) = pts[i]
            dt = t2 - t1
            dx, dy = x2 - x1, y2 - y1
            dist = np.sqrt(dx**2 + dy**2)
            
            if dt > 0:
                v = dist / dt
                seg_velocities.append(v)
                velocities.append(v)
                total_dist += dist
                angles.append(np.arctan2(dy, dx))
        
        path_lengths.append(total_dist)
        
        # Curvature: total path length / straight line distance
        start_pt, end_pt = pts[0][1], pts[-1][1]
        straight_dist = np.sqrt((end_pt[0]-start_pt[0])**2 + (end_pt[1]-start_pt[1])**2)
        if straight_dist > 0: curvatures.append(total_dist / straight_dist)
        
        # Accelerations
        for i in range(1, len(seg_velocities)):
            dv = seg_velocities[i] - seg_velocities[i-1]
            dt_seg = pts[i][0] - pts[i-1][0]
            if dt_seg > 0: accelerations.append(dv / dt_seg)
            
    return {
        'click_dwells': click_dwells, 
        'pause_times': pause_times, 
        'velocities': velocities, 
        'accelerations': accelerations, 
        'curvatures': curvatures, 
        'path_lengths': path_lengths, 
        'angles': [abs(a) for a in angles], # Using absolute for distribution fitting
        'click_frequency': click_count / duration if duration > 0 else 0
    }

In [8]:
COMPREHENSIVE_OUTPUT = '../../data_processed/feature_kmt_dataset_Edge_Hill_University_22/feature_extraction_V4/feature_extraction_V4_comprehensive_88_users.csv'
all_comp_records = []

print(f"Processing comprehensive features for {len(file_list)} users...")

for file_name in file_list:
    user_id = file_name.split('_')[-1].replace('.json', '')
    file_path = os.path.join(RAW_DATA_PATH, file_name)
    
    with open(file_path, 'r') as f:
        data = json.load(f)
        
        for trial_type, label in [('true_data', 1), ('false_data', 0)]:
            trials = data.get(trial_type, {})
            for trial_id, trial_content in trials.items():
                key_events = trial_content.get('key_events', [])
                mouse_events = trial_content.get('mouse_events', [])
                if not key_events: continue
                
                # 1. Keystroke Timings
                raw_dwells, raw_digraphs = get_raw_latencies(key_events)
                raw_pp, raw_rr = get_flight_times(key_events)
                
                # 2. Mouse Metrics
                m_metrics = get_mouse_metrics(mouse_events)
                
                # 3. Build Record
                record = {
                    'user_id': user_id, 
                    'trial_id': trial_id, 
                    'label': label,
                    'click_frequency': m_metrics.get('click_frequency', 0)
                }
                
                # -- Keystroke Feature Sets --
                record.update(calculate_features(remove_outliers(raw_dwells), 'dwell'))
                record.update(calculate_features(remove_outliers(raw_digraphs), 'digraph'))
                record.update(calculate_features(remove_outliers(raw_pp), 'pp'))
                record.update(calculate_features(remove_outliers(raw_rr), 'rr'))
                
                # -- Mouse Feature Sets --
                record.update(calculate_features(remove_outliers(m_metrics.get('click_dwells', [])), 'm_click_dwell'))
                record.update(calculate_features(remove_outliers(m_metrics.get('pause_times', [])), 'm_pause'))
                record.update(calculate_features(remove_outliers(m_metrics.get('velocities', [])), 'm_velocity'))
                record.update(calculate_features(remove_outliers(m_metrics.get('accelerations', [])), 'm_acceleration'))
                record.update(calculate_features(remove_outliers(m_metrics.get('curvatures', [])), 'm_curvature'))
                record.update(calculate_features(remove_outliers(m_metrics.get('path_lengths', [])), 'm_path_length'))
                record.update(calculate_features(remove_outliers(m_metrics.get('angles', [])), 'm_angle'))
                
                all_comp_records.append(record)
    
df_comp = pd.DataFrame(all_comp_records)

# Ensure output directory exists
if not os.path.exists(os.path.dirname(COMPREHENSIVE_OUTPUT)):
    os.makedirs(os.path.dirname(COMPREHENSIVE_OUTPUT))

df_comp.to_csv(COMPREHENSIVE_OUTPUT, index=False)
print(f"Comprehensive features saved to: {COMPREHENSIVE_OUTPUT}")
display(df_comp.head())

Processing comprehensive features for 88 users...
Comprehensive features saved to: ../../data_processed/feature_kmt_dataset_Edge_Hill_University_22/feature_extraction_V4/feature_extraction_V4_comprehensive_88_users.csv


,user_id,trial_id,label,click_frequency,dwell_gamma_shape,dwell_gamma_loc,dwell_gamma_scale,dwell_lognorm_s,dwell_lognorm_loc,dwell_lognorm_scale,...,m_angle_gamma_loc,m_angle_gamma_scale,m_angle_lognorm_s,m_angle_lognorm_loc,m_angle_lognorm_scale,m_angle_best_fit,m_angle_moment_mean,m_angle_moment_std,m_angle_moment_skew,m_angle_moment_kurtosis
0,0001,test_1,1,0.197128,27.332493,0,0.003162,0.194295,0,0.084838,...,0.0,0.318759,0.606597,0,1.080384,Gamma,1.235881,0.504454,-0.259652,0.232500
1,0001,test_2,1,0.238123,36.094959,0,0.002367,0.169500,0,0.084252,...,0.0,0.071115,0.212647,0,1.550251,Log-Normal,1.585673,0.338772,0.430999,-0.572288
2,0001,test_3,1,0.228472,24.226644,0,0.003605,0.209384,0,0.085550,...,0.0,0.359670,0.519995,0,1.590863,Gamma,1.767329,0.712645,0.317383,-0.379099
3,0001,test_4,1,0.187265,27.260723,0,0.003219,0.197891,0,0.086148,...,0.0,0.380654,0.493756,0,1.588652,Gamma,1.775202,0.794115,0.444288,-0.880785
4,0001,test_5,1,0.180207,21.852163,0,0.003766,0.217211,0,0.080422,...,0.0,0.219572,0.366177,0,1.810333,Gamma,1.919011,0.591768,-0.099615,-0.118381
